# 05 — Piper TTS offline: comparativa de voces ES/EN para LastLight-Voice

**Experimento reproducible con Rust/Evcxr** para [`rust-for-ai`](https://github.com/edujbarrios/rust-for-ai), orientado a [`LastLight-Voice`](https://github.com/edujbarrios/LastLight-Voice). Fecha de referencia del catálogo: **11 de noviembre de 2026**.

**Pregunta:** ¿qué par de voces Piper proporciona español + inglés con menos espacio en disco y buen rendimiento de síntesis *realmente offline*?

> **Bilingüe ≠ una voz que habla dos idiomas.** Piper ofrece modelos entrenados por idioma. Aquí seleccionamos **una voz ES y otra EN** y enrutamos el texto según el idioma conocido. No se afirma que un modelo monolingüe haga code-switching o traducción; la detección de idioma queda fuera del experimento.

**Separación entre datos y pruebas:** las cifras de tamaño de la primera tabla son **metadatos publicados en `voices.json`**, no mediciones locales. Los tiempos, RTF y RAM **solo se muestran si se descargan los pesos y se ejecuta Piper en el equipo local**. No hay resultados de inferencia inventados ni se confunde `low` con un modelo necesariamente más pequeño. Los archivos de pesos no deben subirse a Git.

**Procedencia:** [manifest oficial](https://huggingface.co/rhasspy/piper-voices/resolve/main/voices.json) · [modelos](https://huggingface.co/rhasspy/piper-voices) · [demo solicitada](https://piper.ttstool.com/) · [documentación Piper](https://tderflinger.github.io/piper-docs/guides/usage/). Revise el `MODEL_CARD` de cada voz antes de redistribuirla: las licencias pueden variar.

## 0. Preparación, coste de descarga y modo offline

1. En `rust-for-ai`, abre este archivo en `notebooks/` y utiliza el **kernel Rust (Evcxr)** proporcionado por `bash start.sh` o `start.ps1`.
2. Instala el motor Piper en el equipo de pruebas **una vez**, por ejemplo con `python -m pip install piper-tts` dentro de su entorno Python. Se trata de una dependencia **separada** del runtime actual de LastLight-Voice. Comprueba `piper --help`. Si el ejecutable no está en `PATH`, define `PIPER_BIN` con su ruta.
3. La **primera descarga** es opcional y explícita: inicia Jupyter con `PIPER_PREPARE=1` para obtener los dos modelos elegidos por defecto desde Hugging Face. En macOS/Linux, por ejemplo: `PIPER_PREPARE=1 bash start.sh`. En PowerShell: `$env:PIPER_PREPARE='1'; .\start.ps1`.
4. Para **comprobar el modo offline**, inicia después Jupyter con `PIPER_PREPARE=0` (o sin esa variable), desconecta la red y ejecuta todas las celdas. No se necesita red si los modelos y Piper están ya instalados.

Configura `PIPER_MODELS_DIR` para cambiar el directorio de modelos; por defecto se usa `std::env::temp_dir()/lastlight-piper-models`, externo a Git. Los WAV y el CSV van al subdirectorio `benchmarks/`. En una instalación real, traslada **los modelos, sus JSON, el motor Piper y los recursos fonéticos de eSpeak-NG** al dispositivo objetivo *antes* de desconectarlo.

**Interpretación del tamaño:** se calcula el peso de los modelos `.onnx` y su configuración JSON, **no** el tamaño del binario de Piper, dependencias de ONNX Runtime, eSpeak-NG, bibliotecas del SO, RAM ni instalación completa. `MB` decimal = 1.000.000 bytes; `MiB` = 1.048.576 bytes.

## 1. Catálogo contrastado: bytes reales declarados en `voices.json`

Esta es una **selección reproducible de siete voces**, no una búsqueda exhaustiva de todo el catálogo. Incluye opciones ultra-ligeras, low y medium para poder comparar tamaño frente a rendimiento. Los números de bytes se transcribieron del manifiesto oficial. No deducimos calidad acústica a partir del nombre del nivel.

In [ ]:
#[derive(Clone, Copy)]
struct Voice {
    id: &'static str,
    language: &'static str,
    tier: &'static str,
    hf_dir: &'static str,
    model_bytes: u64,
    config_bytes: u64,
}
let voices: [Voice; 7] = [
    Voice { id: "es_ES-carlfm-x_low", language: "es", tier: "x_low", hf_dir: "es/es_ES/carlfm/x_low", model_bytes: 28_130_791, config_bytes: 4_159 },
    Voice { id: "es_ES-mls_9972-low", language: "es", tier: "low", hf_dir: "es/es_ES/mls_9972/low", model_bytes: 63_104_526, config_bytes: 4_159 },
    Voice { id: "es_ES-davefx-medium", language: "es", tier: "medium", hf_dir: "es/es_ES/davefx/medium", model_bytes: 63_221_984, config_bytes: 5_458 },
    Voice { id: "es_ES-sharvard-medium", language: "es", tier: "medium", hf_dir: "es/es_ES/sharvard/medium", model_bytes: 76_753_841, config_bytes: 5_486 },
    Voice { id: "en_US-amy-low", language: "en", tier: "low", hf_dir: "en/en_US/amy/low", model_bytes: 63_104_526, config_bytes: 4_164 },
    Voice { id: "en_US-ryan-low", language: "en", tier: "low", hf_dir: "en/en_US/ryan/low", model_bytes: 63_104_526, config_bytes: 4_165 },
    Voice { id: "en_US-lessac-medium", language: "en", tier: "medium", hf_dir: "en/en_US/lessac/medium", model_bytes: 63_201_294, config_bytes: 4_885 },
];
println!("{:<26} {:<3} {:<7} {:>12} {:>12}", "MODELO", "IDI", "NIVEL", "MB decimal", "MiB binario");
for v in &voices {
    let total = v.model_bytes + v.config_bytes;
    println!("{:<26} {:<3} {:<7} {:>12.2} {:>12.2}", v.id, v.language, v.tier, total as f64 / 1_000_000.0, total as f64 / 1_048_576.0);
}
let es_min = voices.iter().filter(|v| v.language == "es").min_by_key(|v| v.model_bytes + v.config_bytes).unwrap();
let en_min = voices.iter().filter(|v| v.language == "en").min_by_key(|v| v.model_bytes + v.config_bytes).unwrap();
let min_bytes = es_min.model_bytes + es_min.config_bytes + en_min.model_bytes + en_min.config_bytes;
println!("\nPar mínimo en esta selección: {} + {} = {:.2} MB / {:.2} MiB (solo modelos+JSON)", es_min.id, en_min.id, min_bytes as f64 / 1_000_000.0, min_bytes as f64 / 1_048_576.0);

## 2. Preparar los pesos (solo cuando `PIPER_PREPARE=1`)

Para impedir descargas accidentales en un entorno desconectado, **no se hace ninguna solicitud HTTP por defecto**. Si `PIPER_PREPARE=1`, se usa `curl` desde Rust para descargar `.onnx`, `.onnx.json` y `MODEL_CARD` (licencia/procedencia) de cada voz seleccionada; se descargan primero a `.part` y se comprueba el tamaño publicado antes de renombrar. **La comprobación de tamaño no equivale a una validación criptográfica**. Para un paquete distribuible conviene añadir SHA-256 propio y verificación de la cadena de suministro.

Los dos modelos predeterminados ocupan unos 87,02 MiB (pesos y JSON). Puede ampliar `selected_ids` si desea probar más voces. Los modelos ya guardados se reutilizan, pero únicamente si coinciden los tamaños esperados.

In [ ]:
use std::path::{Path, PathBuf};
use std::process::{Command, Stdio};
use std::io::Write;
use std::time::Instant;

let model_dir = std::env::var_os("PIPER_MODELS_DIR")
    .map(PathBuf::from)
    .unwrap_or_else(|| std::env::temp_dir().join("lastlight-piper-models"));
std::fs::create_dir_all(&model_dir).expect("No se pudo crear el directorio de modelos");
let selected_ids = [
    "es_ES-carlfm-x_low", "en_US-amy-low",
];
let prepare_downloads = std::env::var("PIPER_PREPARE").map(|x| x == "1").unwrap_or(false);
println!("Directorio: {} | Descarga voluntaria: {}", model_dir.display(), prepare_downloads);
for v in voices.iter().filter(|v| selected_ids.contains(&v.id)) {
    let files = [
        (format!("{}.onnx", v.id), v.model_bytes, false),
        (format!("{}.onnx.json", v.id), v.config_bytes, false),
        (String::from("MODEL_CARD"), 0_u64, true),
    ];
    for (remote_name, expected_bytes, is_card) in files {
        let local_name = if is_card { format!("{}.MODEL_CARD", v.id) } else { remote_name.clone() };
        let dest = model_dir.join(&local_name);
        let found = std::fs::metadata(&dest).map(|x| x.len()).unwrap_or(0);
        let valid = if is_card { found > 0 } else { found == expected_bytes };
        if valid { continue; }
        if !prepare_downloads {
            if !is_card { println!("PENDIENTE: {} (usa PIPER_PREPARE=1 con Internet)", local_name); }
            continue;
        }
        let partial = model_dir.join(format!("{}.part", local_name));
        let url = format!("https://huggingface.co/rhasspy/piper-voices/resolve/main/{}/{}", v.hf_dir, remote_name);
        let status = Command::new("curl")
            .args(["--fail", "--location", "--silent", "--show-error", "--retry", "2", "--connect-timeout", "20", "--max-time", "300", "--output"])
            .arg(&partial).arg(&url).status();
        let ok = status.map(|s| s.success()).unwrap_or(false);
        let received = std::fs::metadata(&partial).map(|x| x.len()).unwrap_or(0);
        if ok && (is_card && received > 0 || !is_card && received == expected_bytes) {
            if let Err(e) = std::fs::rename(&partial, &dest) { eprintln!("No se pudo mover {}: {}", local_name, e); }
            else { println!("OK: {} ({:.2} MiB)", local_name, received as f64 / 1_048_576.0); }
        } else {
            eprintln!("ERROR de descarga/tamaño: {} ({} bytes; esperados {})", local_name, received, expected_bytes);
            let _ = std::fs::remove_file(&partial);
        }
    }
}
let ready_ids: Vec<&str> = voices.iter().filter(|v| selected_ids.contains(&v.id)).filter(|v| {
    let model = model_dir.join(format!("{}.onnx", v.id));
    let config = model_dir.join(format!("{}.onnx.json", v.id));
    std::fs::metadata(model).map(|m| m.len()).unwrap_or(0) == v.model_bytes &&
    std::fs::metadata(config).map(|m| m.len()).unwrap_or(0) == v.config_bytes
}).map(|v| v.id).collect();
println!("Preparadas para probar: {} / {}", ready_ids.len(), selected_ids.len());

## 3. Benchmark offline real: latencia, RTF y memoria RSS máxima

Se ejecuta `piper --model RUTA_LOCAL.onnx --output_file salida.wav` con el texto por **stdin**. No se solicita ningún ID de voz remoto. La salida WAV se verifica y se lee la duración desde su encabezado PCM; no se usa un número de segundos estimado.

Métricas:
- **Latencia de síntesis (ms):** tiempo de reloj hasta que termina el proceso y se escribe el WAV. Incluye lanzamiento del ejecutable y carga del modelo. **No mide la latencia de reproducción por altavoces**.
- **RTF = segundos de cálculo / segundos de audio.** Menor es mejor. Se repite cada texto dos veces en **procesos nuevos**; esto **no** mide inferencia con modelo persistente en RAM. Un RTF de `0.2` significa 5× más rápido que la reproducción, *en ese hardware*.
- **Pico de RAM (MiB):** en Linux, si está disponible GNU `/usr/bin/time`, registra `maxrss` en KiB. En otros sistemas figura `N/D`; no se inventa una estimación.

Se usan frases cortas y largas en cada idioma, representativas de mensajes de emergencia de LastLight; generan WAV de muestra para escuchar. Para comparaciones concluyentes es preciso repetir en el **hardware objetivo** (por ejemplo Raspberry Pi) y con idéntica carga/CPU. **No es una prueba objetiva de calidad o inteligibilidad.**

In [ ]:
fn wav_duration_seconds(path: &Path) -> Result<f64, String> {
    let b = std::fs::read(path).map_err(|e| e.to_string())?;
    if b.len() < 44 || &b[0..4] != b"RIFF" || &b[8..12] != b"WAVE" { return Err(String::from("WAV RIFF no válido")); }
    let mut p: usize = 12;
    let mut rate: u32 = 0;
    let mut align: u16 = 0;
    let mut data_bytes: usize = 0;
    while p + 8 <= b.len() {
        let size = u32::from_le_bytes(b[p+4..p+8].try_into().unwrap()) as usize;
        let start = p + 8;
        if start + size > b.len() { return Err(String::from("Chunk WAV truncado")); }
        if &b[p..p+4] == b"fmt " && size >= 16 {
            let format = u16::from_le_bytes(b[start..start+2].try_into().unwrap());
            if format != 1 { return Err(String::from("Este lector espera WAV PCM sin comprimir")); }
            rate = u32::from_le_bytes(b[start+4..start+8].try_into().unwrap());
            align = u16::from_le_bytes(b[start+12..start+14].try_into().unwrap());
        }
        if &b[p..p+4] == b"data" { data_bytes = size; break; }
        p = start + size + (size % 2);
    }
    if rate == 0 || align == 0 || data_bytes == 0 { return Err(String::from("Falta fmt/data o audio vacío")); }
    Ok(data_bytes as f64 / (rate as f64 * align as f64))
}
#[derive(Clone)]
struct Measurement {
    id: String,
    language: String,
    case_name: String,
    run: u32,
    time_ms: f64,
    duration_s: f64,
    rtf: f64,
    max_rss_mib: Option<f64>,
    wav_path: String,
}
let piper_bin = std::env::var("PIPER_BIN").unwrap_or_else(|_| String::from("piper"));
let piper_available = Command::new(&piper_bin).arg("--help").output().is_ok();
let output_dir = model_dir.join("benchmarks");
std::fs::create_dir_all(&output_dir).expect("No se pudo crear benchmarks/");
let benchmark_texts: [(&str, &str, &str); 4] = [
    ("es", "corto", "Aviso: hierve el agua antes de beberla."),
    ("es", "largo", "LastLight funciona sin conexión. Si no hay electricidad ni acceso a internet, conserva agua potable, consulta la información local verificada y sigue las instrucciones oficiales de seguridad. No difundas datos que no estén confirmados."),
    ("en", "corto", "Alert: boil water before drinking it."),
    ("en", "largo", "LastLight works without an internet connection. If power and network access are unavailable, store safe drinking water, consult verified local information, and follow official safety instructions. Do not share unconfirmed claims."),
];
let mut measurements: Vec<Measurement> = Vec::new();
if !piper_available {
    println!("SIN PRUEBAS DE INFERENCIA: falta Piper ({}). Instala piper-tts y define PIPER_BIN si hace falta.", piper_bin);
} else if ready_ids.is_empty() {
    println!("SIN PRUEBAS DE INFERENCIA: no hay voces locales completas. Descárgalas explícitamente con PIPER_PREPARE=1.");
} else {
    println!("Benchmark offline de {} voces en {} ({} CPUs lógicas)", ready_ids.len(), std::env::consts::OS, std::thread::available_parallelism().map(|n| n.get()).unwrap_or(1));
    for v in voices.iter().filter(|v| ready_ids.contains(&v.id)) {
        for (language, case_name, phrase) in benchmark_texts.iter().copied().filter(|t| t.0 == v.language) {
            for run in 1..=2_u32 {
                let wav = output_dir.join(format!("{}_{}_{}.wav", v.id, case_name, run));
                let rss_file = output_dir.join(format!("{}_{}_{}.rss", v.id, case_name, run));
                let capture_rss = cfg!(target_os = "linux") && Path::new("/usr/bin/time").is_file();
                let mut command = if capture_rss { Command::new("/usr/bin/time") } else { Command::new(&piper_bin) };
                if capture_rss { command.args(["-f", "%M", "-o"]).arg(&rss_file).arg(&piper_bin); }
                let model = model_dir.join(format!("{}.onnx", v.id));
                command.arg("--model").arg(&model).arg("--output_file").arg(&wav)
                    .stdin(Stdio::piped()).stdout(Stdio::null()).stderr(Stdio::piped());
                let t0 = Instant::now();
                let result = match command.spawn() {
                    Ok(mut child) => {
                        let written = child.stdin.take().map(|mut input| input.write_all(phrase.as_bytes()));
                        if let Some(Err(e)) = written { eprintln!("No se pudo enviar el texto: {}", e); }
                        child.wait_with_output()
                    },
                    Err(e) => Err(e),
                };
                let elapsed = t0.elapsed().as_secs_f64() * 1000.0;
                match result {
                    Ok(out) if out.status.success() => match wav_duration_seconds(&wav) {
                        Ok(duration) if duration > 0.0 => {
                            let rss = if capture_rss { std::fs::read_to_string(&rss_file).ok().and_then(|s| s.trim().parse::<f64>().ok()).map(|kib| kib / 1024.0) } else { None };
                            println!("{:<26} {:<5} rep={} tiempo={:>8.0}ms audio={:>6.2}s RTF={:>5.3} RAM={}", v.id, case_name, run, elapsed, duration, elapsed / (1000.0 * duration), rss.map(|r| format!("{:.1} MiB", r)).unwrap_or_else(|| String::from("N/D")));
                            measurements.push(Measurement { id: String::from(v.id), language: String::from(language), case_name: String::from(case_name), run, time_ms: elapsed, duration_s: duration, rtf: elapsed/(1000.0*duration), max_rss_mib: rss, wav_path: wav.display().to_string() });
                        },
                        Ok(_) => eprintln!("WAV vacío: {}", wav.display()),
                        Err(e) => eprintln!("WAV no válido {}: {}", wav.display(), e),
                    },
                    Ok(out) => eprintln!("Piper falló para {}: {}", v.id, String::from_utf8_lossy(&out.stderr)),
                    Err(e) => eprintln!("No se pudo lanzar Piper para {}: {}", v.id, e),
                }
            }
        }
    }
}
println!("Mediciones REALES terminadas: {} (0 significa que NO se ha realizado inferencia).", measurements.len());

## 4. Guardar los resultados y comparar por tamaño, tiempo y calidad

Se genera `piper_benchmark.csv` con resultados medidos. El par más ligero se obtiene **solo de los metadatos**; el par más rápido requiere **mediciones locales**. Mantén separados ambos conceptos.

**Evaluación de escucha recomendada:** abre los WAV generados desde el panel de ficheros y puntúa para cada voz (de 1 a 5) **inteligibilidad**, **naturalidad** y **pronunciación** de cifras, siglas y topónimos. Intenta hacerlo sin conocer el nombre del modelo. En LastLight-Voice la inteligibilidad de avisos de seguridad debe pesar más que la expresividad. No se asignan puntuaciones subjetivas por defecto.

In [ ]:
let csv_path = output_dir.join("piper_benchmark.csv");
let mut csv = String::from("voice,language,case,run,time_ms,audio_s,rtf,max_rss_mib,wav_path\n");
for m in &measurements {
    csv.push_str(&format!("{},{},{},{},{:.3},{:.3},{:.5},{},{}\n", m.id, m.language, m.case_name, m.run, m.time_ms, m.duration_s, m.rtf, m.max_rss_mib.map(|x| format!("{:.2}", x)).unwrap_or_default(), m.wav_path));
}
std::fs::write(&csv_path, csv).expect("No se pudo escribir el CSV");
println!("CSV: {}", csv_path.display());
println!("\nRendimiento promedio observado (no equivale a una nota de calidad):");
for v in &voices {
    let obs: Vec<&Measurement> = measurements.iter().filter(|m| m.id == v.id).collect();
    if obs.is_empty() { continue; }
    let mean_rtf = obs.iter().map(|m| m.rtf).sum::<f64>() / obs.len() as f64;
    let short_ms: Vec<f64> = obs.iter().filter(|m| m.case_name == "corto").map(|m| m.time_ms).collect();
    let short_latency = if short_ms.is_empty() { 0.0 } else { short_ms.iter().sum::<f64>() / short_ms.len() as f64 };
    let max_rss = obs.iter().filter_map(|m| m.max_rss_mib).fold(0.0_f64, f64::max);
    println!("{:<26} {:>7.2} MiB | RTF medio={:>5.3} | corto={:>7.0} ms | RAM={}", v.id, (v.model_bytes+v.config_bytes) as f64 / 1_048_576.0, mean_rtf, short_latency, if max_rss > 0.0 { format!("{:.1} MiB", max_rss) } else { String::from("N/D") });
}
println!("\nMás ligero de la selección: {} + {} = {:.2} MiB", es_min.id, en_min.id, min_bytes as f64/1_048_576.0);
if !measurements.is_empty() {
    for lang in ["es", "en"] {
        let ranked = voices.iter().filter(|v| v.language == lang).filter_map(|v| {
            let obs: Vec<&Measurement> = measurements.iter().filter(|m| m.id == v.id).collect();
            if obs.is_empty() { None } else { Some((v.id, obs.iter().map(|m| m.rtf).sum::<f64>() / obs.len() as f64)) }
        }).min_by(|a,b| a.1.partial_cmp(&b.1).unwrap_or(std::cmp::Ordering::Equal));
        if let Some((name, rtf)) = ranked { println!("RTF medio más bajo entre los medidos en {}: {} ({:.3})", lang, name, rtf); }
    }
}

## 5. Decisión responsable: calidad mínima y presupuesto

Rellena con tus propias notas `quality_votes` después de escuchar los WAV (**0 = no evaluada; 1–5 = media de inteligibilidad, naturalidad y pronunciación**). Elige una nota **mínima de 3** en ambos idiomas. Si una voz no ha sido escuchada, **no se recomienda automáticamente** como «la mejor». La selección de abajo busca el **menor tamaño total** entre las voces con nota suficiente y, si existe un empate en bytes, conserva el orden del catálogo; compara los RTF medidos para resolver empates. El resultado es una **regla de decisión explícita**, no una clasificación universal.

In [ ]:
let quality_votes: [(&str, u8); 7] = [
    ("es_ES-carlfm-x_low", 0),
    ("es_ES-mls_9972-low", 0),
    ("es_ES-davefx-medium", 0),
    ("es_ES-sharvard-medium", 0),
    ("en_US-amy-low", 0),
    ("en_US-ryan-low", 0),
    ("en_US-lessac-medium", 0),
];
let budget_mib: f64 = 140.0;
let qualified: Vec<&Voice> = voices.iter().filter(|v| {
    quality_votes.iter().find(|(id, _)| *id == v.id).map(|(_, score)| *score >= 3).unwrap_or(false)
}).collect();
let mut pairs: Vec<(&Voice, &Voice, u64)> = Vec::new();
for es in qualified.iter().copied().filter(|v| v.language == "es") {
    for en in qualified.iter().copied().filter(|v| v.language == "en") {
        let bytes = es.model_bytes + es.config_bytes + en.model_bytes + en.config_bytes;
        if (bytes as f64 / 1_048_576.0) <= budget_mib { pairs.push((es,en,bytes)); }
    }
}
pairs.sort_by_key(|p| p.2);
match pairs.first() {
    Some((es,en,bytes)) => println!("MENOR PAQUETE APROBADO: {} + {} = {:.2} MiB (voces+JSON; falta medir instalación completa)", es.id,en.id,*bytes as f64 / 1_048_576.0),
    None => println!("Sin selección validada: puntúa al menos una voz ES y una EN con >=3 y ajusta presupuesto ({} MiB).", budget_mib),
};

## 6. Integración experimental con LastLight-Voice, sin tocar su API

En la versión actual de [`LastLight-Voice`](https://github.com/edujbarrios/LastLight-Voice) los backends disponibles son **eSpeak NG** y **Dummy**; **Piper no está aún incorporado** en `SpeechEngine`. Este ejemplo en **Python** (solo como documentación, **no es una celda Rust ejecutable**) muestra el puente mínimo para probar archivos WAV offline usando el **par seleccionado**.

```python
from pathlib import Path
import subprocess

MODELS = Path('/ruta/local/a/lastlight-piper-models')  # cambia esta ruta
PIPER = 'piper'  # o ruta absoluta al ejecutable local
VOICES = {'es': 'es_ES-carlfm-x_low', 'en': 'en_US-amy-low'}

def synthesize_offline(text: str, language: str, output: str | Path) -> Path:
    """Sin DNS, sin requests, sin fallback silencioso a una descarga."""
    name = VOICES[language]  # idioma seleccionado explícitamente por la app
    model = MODELS / f'{name}.onnx'
    config = MODELS / f'{name}.onnx.json'
    if not model.is_file() or not config.is_file():
        raise FileNotFoundError(f'Modelo offline ausente: {name}')
    destination = Path(output)
    subprocess.run([PIPER, '--model', str(model), '--output_file', str(destination)],
                   input=text, text=True, check=True, timeout=120)
    return destination

synthesize_offline('El conocimiento sigue disponible sin red.', 'es', 'alerta_es.wav')
synthesize_offline('Knowledge remains available without a network.', 'en', 'alert_en.wav')
```

Para conectar Piper al `SpeechEngine` existente hay que implementar el contrato `lastlight_voice.backends.base.TTSBackend` (`available`, `capabilities`, `voices`, `speak`, `synthesize`, `stop`) y convertir el WAV a `AudioBuffer.from_wav_bytes`; **no** basta con declarar `backend='piper'`, pues ese nombre aún no existe en `_resolve_backend`. Mantén eSpeak NG como fallback explícito de muy bajo consumo; evítese el fallback automático silencioso en un sistema de emergencias.

## 7. Interpretación inicial y límites de la prueba

**Tamaño verificable del par ultraligero de esta selección:** `es_ES-carlfm-x_low` + `en_US-amy-low` equivale a aproximadamente **91,24 MB decimales / 87,02 MiB** sumando ambos `.onnx` y JSON; `en_US-ryan-low` ocupa virtualmente lo mismo y permite elegir una voz inglesa masculina en vez de Amy. Estas cifras proceden del catálogo y no prueban por sí solas que la salida sea la más inteligible, rápida ni que el sistema completo ocupe menos de 100 MB.

**No confundir:** tamaño en disco ≠ tamaño en memoria; `low` ≠ menor tamaño de forma garantizada; `medium` ≠ mejor pronunciación de cifras comprobada; `RTF` depende del CPU, longitud del texto, temperatura y modelo de ejecución. La demo `piper.ttstool.com` permite una escucha orientativa, pero los WAV de esta prueba y la escucha ciega son los que deben decidir sobre el dispositivo real. El usuario final **no** debería necesitar conexión ni descarga durante la síntesis.

**Lista de comprobación para producción:** probar frases con números y ubicaciones; medir CPU/RAM en Raspberry Pi si aplica; comprobar los 2 idiomas al desconectar la red; fijar el hash y licencia por voz; medir tamaño *instalado* de Piper+ONNX Runtime+eSpeak; impedir descargas implícitas; valorar un proceso persistente si la latencia de arranque penaliza; no copiar la carpeta de caché a Git.

Referencias: [Piper voices](https://huggingface.co/rhasspy/piper-voices), [manifest `voices.json`](https://huggingface.co/rhasspy/piper-voices/resolve/main/voices.json), [Piper usage](https://tderflinger.github.io/piper-docs/guides/usage/), [rust-for-ai](https://github.com/edujbarrios/rust-for-ai), [LastLight-Voice](https://github.com/edujbarrios/LastLight-Voice).

## 8. Prueba de escucha real: dos voces, cuatro MP3

Estas celdas adicionales sintetizan **dos textos en español** con `es_ES-carlfm-x_low` y **dos textos en inglés** con `en_US-amy-low`. La prueba incluye un aviso corto (cifras/horas) y otro texto más largo, para juzgar **inteligibilidad y pronunciación**. No se traduce ni se cambia de voz dentro de una misma frase.

**Requisitos:** haber ejecutado las celdas anteriores (variables `model_dir`, `ready_ids`, `piper_bin`), tener Piper instalado y ambos modelos ONNX+JSON descargados previamente y disponer del ejecutable `ffmpeg` en `PATH`. La síntesis y conversión se hacen **sin Internet**. Piper genera WAV y **FFmpeg convierte WAV a MP3** (Piper no genera MP3 directamente). Si falta algún requisito, se informa y se omite esa muestra: jamás se simula audio.

Los audios se guardan en `piper_tts_samples/` relativo al **directorio de trabajo actual** de Jupyter. Puedes definir `PIPER_MP3_DIR` para cambiarlo. Se mantienen WAV y MP3 para escuchar y diagnosticar posibles problemas. **No subas audios ni pesos a Git sin revisar su licencia.**

In [ ]:
let listening_texts: [(&str, &str, &str, &str); 4] = [
    (
        "es_ES-carlfm-x_low", "es_corto", "es",
        "Atención. Son las ocho y treinta. La temperatura es de veintidós grados. Hierve el agua durante un minuto antes de beberla.",
    ),
    (
        "es_ES-carlfm-x_low", "es_largo", "es",
        "Hola, soy LastLight Voice. Puedo leer información útil aunque no haya conexión a Internet. Si se interrumpe el suministro eléctrico, conserva agua potable, mantén cargada una linterna y consulta únicamente instrucciones de seguridad verificadas. Ante una emergencia, sigue las indicaciones de los servicios oficiales.",
    ),
    (
        "en_US-amy-low", "en_short", "en",
        "Attention. It is eight thirty. The temperature is twenty two degrees. Boil water for one minute before drinking it.",
    ),
    (
        "en_US-amy-low", "en_long", "en",
        "Hello, I am LastLight Voice. I can read useful information even without an internet connection. If the power goes out, keep safe drinking water available, prepare a flashlight, and consult only verified safety instructions. During an emergency, follow guidance from official services.",
    ),
];
let mp3_output_dir = std::env::var_os("PIPER_MP3_DIR")
    .map(std::path::PathBuf::from)
    .unwrap_or_else(|| std::env::current_dir().expect("Sin directorio de trabajo").join("piper_tts_samples"));
std::fs::create_dir_all(&mp3_output_dir).expect("No se pudo crear piper_tts_samples");
let has_ffmpeg = std::process::Command::new("ffmpeg")
    .arg("-version").output().map(|o| o.status.success()).unwrap_or(false);
let mut listening_results: Vec<(String, String, String, u64, Option<f64>)> = Vec::new();
println!("Destino WAV/MP3: {}", mp3_output_dir.display());
if !piper_available {
    println!("NO GENERADO: ejecutable Piper ausente; instala piper-tts y vuelve a ejecutar.");
} else if !has_ffmpeg {
    println!("NO GENERADO: falta ffmpeg en PATH; instálalo para convertir WAV a MP3.");
} else {
    for (voice_id, sample_id, language, phrase) in listening_texts {
        if !ready_ids.contains(&voice_id) {
            println!("SALTADO: {} (falta ONNX/JSON descargado)", voice_id);
            continue;
        }
        let model_file = model_dir.join(format!("{}.onnx", voice_id));
        let wav_file = mp3_output_dir.join(format!("{}.wav", sample_id));
        let mp3_file = mp3_output_dir.join(format!("{}.mp3", sample_id));
        let _ = std::fs::remove_file(&wav_file);
        let _ = std::fs::remove_file(&mp3_file);
        let start = std::time::Instant::now();
        let mut child = match std::process::Command::new(&piper_bin)
            .arg("--model").arg(&model_file)
            .arg("--output_file").arg(&wav_file)
            .stdin(std::process::Stdio::piped())
            .stdout(std::process::Stdio::null())
            .stderr(std::process::Stdio::piped())
            .spawn()
        {
            Ok(child) => child,
            Err(err) => { eprintln!("Piper no arranca con {}: {}", voice_id, err); continue; }
        };
        let input_result = match child.stdin.take() {
            Some(mut input) => std::io::Write::write_all(&mut input, phrase.as_bytes()),
            None => Err(std::io::Error::new(std::io::ErrorKind::BrokenPipe, "stdin de Piper cerrado")),
        };
        if let Err(err) = input_result { eprintln!("Error de entrada de Piper: {}", err); }
        let synth_result = child.wait_with_output();
        let synthesis_ms = start.elapsed().as_secs_f64() * 1000.0;
        match synth_result {
            Ok(result) if result.status.success() && wav_file.exists() => {},
            Ok(result) => { eprintln!("Fallo en Piper {}: {}", voice_id, String::from_utf8_lossy(&result.stderr)); continue; },
            Err(err) => { eprintln!("Error al esperar Piper: {}", err); continue; },
        }
        let convert = std::process::Command::new("ffmpeg")
            .args(["-hide_banner", "-loglevel", "error", "-y", "-i"])
            .arg(&wav_file)
            .args(["-vn", "-ac", "1", "-c:a", "libmp3lame", "-b:a", "96k"])
            .arg(&mp3_file)
            .output();
        match convert {
            Ok(result) if result.status.success() => {},
            Ok(result) => { eprintln!("Fallo al convertir {} a MP3: {}", sample_id, String::from_utf8_lossy(&result.stderr)); continue; },
            Err(err) => { eprintln!("No se pudo ejecutar ffmpeg: {}", err); continue; },
        }
        let mp3_bytes = std::fs::metadata(&mp3_file).map(|m| m.len()).unwrap_or(0);
        if mp3_bytes < 1000 { eprintln!("MP3 vacío o sospechosamente pequeño: {}", mp3_file.display()); continue; }
        let duration = std::process::Command::new("ffprobe")
            .args(["-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1"])
            .arg(&mp3_file).output().ok()
            .filter(|o| o.status.success())
            .and_then(|o| String::from_utf8(o.stdout).ok())
            .and_then(|s| s.trim().parse::<f64>().ok());
        println!("MP3 REAL: {} | {} | {:.1} KiB | duración {} | síntesis {:.0} ms", sample_id, language, mp3_bytes as f64 / 1024.0, duration.map(|x| format!("{:.2} s", x)).unwrap_or_else(|| String::from("no disponible")), synthesis_ms);
        println!("  Texto leído: {}", phrase);
        println!("  Archivo: {}", mp3_file.display());
        listening_results.push((voice_id.to_string(), sample_id.to_string(), mp3_file.display().to_string(), mp3_bytes, duration));
    }
}
println!("Total MP3 generados: {}/4", listening_results.len());

## 9. Comprobar los MP3 generados y generar índice

Esta celda imprime el inventario, verifica que cada MP3 exista y crea `INDICE_MP3.md` con enlaces a los audios desde el directorio de salida. **Revisa la salida**: si indica `0/4`, no hay MP3 reales (faltan modelos, Piper o FFmpeg). Los enlaces se podrán abrir desde el explorador de archivos de JupyterLab/VS Code.

Para comparar los dos modelos de manera justa, escucha `es_corto.mp3` y `en_short.mp3` para detectar fallos en cifras, y `es_largo.mp3`/`en_long.mp3` para valorar claridad y fatiga auditiva. Los cuatro audios no reemplazan una evaluación con hablantes nativos.

In [ ]:
let mut index_md = String::from("# Audios de prueba LastLight Voice — Piper TTS\n\n");
index_md.push_str("Archivos generados localmente con Piper, convertidos a MP3 mediante FFmpeg.\n\n");
index_md.push_str("| Modelo | Prueba | Tamaño | Audio |\n|---|---|---:|---|\n");
for (voice_id, sample_id, full_path, bytes, duration) in &listening_results {
    let name = std::path::Path::new(full_path).file_name().and_then(|n| n.to_str()).unwrap_or("audio.mp3");
    if !std::path::Path::new(full_path).is_file() { eprintln!("Archivo desaparecido: {}", full_path); continue; }
    index_md.push_str(&format!("| `{}` | `{}` | {:.1} KiB | [{}]({}) |\n", voice_id, sample_id, *bytes as f64 / 1024.0, name, name));
    println!("LISTO: {} — {:.1} KiB — {}", name, *bytes as f64 / 1024.0, duration.map(|x| format!("{:.2} s", x)).unwrap_or_else(|| String::from("duración no disponible")));
}
if listening_results.is_empty() {
    index_md.push_str("\n**No se generaron MP3.** Comprueba que `piper`, `ffmpeg` y los modelos locales están disponibles.\n");
}
let index_path = mp3_output_dir.join("INDICE_MP3.md");
std::fs::write(&index_path, index_md).expect("Error al guardar el índice de MP3");
println!("Índice guardado en: {}", index_path.display());
println!("Resumen: {} MP3 válidos de 4 muestras previstas", listening_results.len());

## 10. Escuchar y elegir

1. Abre la carpeta `piper_tts_samples` que imprimió la celda anterior y reproduce **`es_corto.mp3`, `es_largo.mp3`, `en_short.mp3`, `en_long.mp3`**. Si estás en JupyterLab, puedes usar el explorador de archivos; si utilizas VS Code, abre los archivos desde su ubicación.
2. Puntúa **inteligibilidad**, **números y nombres**, **naturalidad**, y **claridad de avisos de emergencia**, de 1 a 5. No extrapoles la calidad de una voz a otra, ni el tamaño de un archivo MP3 al tamaño en RAM del modelo.
3. Si la voz española `carlfm-x_low` no se entiende bien, repite la prueba sustituyendo su ID por `es_ES-mls_9972-low` (previamente descargada en la sección 2). El código usará el modelo indicado en cada entrada.

**Audios de referencia publicados por Piper:** [CarlFM, español](https://huggingface.co/rhasspy/piper-voices/blob/main/es/es_ES/carlfm/x_low/samples/speaker_0.mp3) y [Amy, inglés](https://huggingface.co/rhasspy/piper-voices/blob/main/en/en_US/amy/low/samples/speaker_0.mp3). Estos audios públicos sirven para una escucha rápida, **pero no son los textos ni los MP3 que genera este notebook**.